# 🔑 Solutions — go or nogo? — the real neuron, step by step

One worked answer per exercise. There is usually more than one correct
way to write these — if your self-check printed ✅, yours is right too.

Read this *after* an attempt: seeing the answer teaches far less than
getting stuck did.

In [ ]:
# Run this first. It loads two neurons' recordings from the lecture —
# from data/ if you downloaded them, otherwise straight from GitHub.
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab05/data/"
DATA = Path("data")            # same layout as a uv-start project
DATA.mkdir(exist_ok=True)
NEURON_1 = DATA / "neuron_0588_20180828_5_lecture.csv"
NEURON_2 = DATA / "neuron_0406_20180620_8_lecture.csv"
for path in [NEURON_1, NEURON_2]:
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + path.name, path)

# Columns: 0 frame, 1 dff, 2 go, 3 nogo, 4 reward, 5 lick
dff = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=1)
go  = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=2)

FRAME_RATE = 10.8   # frames per second
print("dff:", dff)
print("go: ", go)

## Solution — Step 1, 🔬 write it: load nogo, `n_frames`, `minutes`

In [ ]:
nogo = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=3)
n_frames = dff.size
minutes = round(n_frames / FRAME_RATE / 60, 1)
print(n_frames, "frames =", minutes, "minutes")

## Solution — Step 2, 🔬 write it: `nogo_on_mean`

In [ ]:
nogo_on_mean = round(dff[nogo == 1].mean(), 3)
print("mean while nogo is on:", nogo_on_mean)

In [ ]:
# DEMO - the frame numbers where go is on
go_on = np.where(go == 1)[0]
print(go_on[:20])
print(len(go_on), "frames with go on")

## Solution — Step 3, 🔬 write it: `nogo_on`

In [ ]:
nogo_on = np.where(nogo == 1)[0]
print(nogo_on[:20])

In [ ]:
# DEMO - the first frame of each go trial
go_starts = go_on[::9]
print(go_starts[:5])
print(len(go_starts), "go trials")

## Solution — Step 4, 🔬 write it: `nogo_starts`

In [ ]:
nogo_starts = nogo_on[::9]
print(len(nogo_starts), "nogo trials; the first starts at frame", nogo_starts[0])

## Solution — Step 5, 🔬 write it: `second_window`

In [ ]:
start = go_starts[1]
second_window = dff[start - 10 : start + 30]
print(second_window.size, "frames")

In [ ]:
# DEMO - every go window, stacked into a grid
go_trials = []
for start in go_starts:
    go_trials.append(dff[start - 10 : start + 30])
go_trials = np.array(go_trials)

print(go_trials.shape)
print("one trial:              ", go_trials[0].shape)
print("one moment, every trial:", go_trials[:, 10].shape)

## Solution — Step 6, 🐞 broken: the trial at frame 2

In [ ]:
nogo_trials = []
for start in nogo_starts:
    if start >= 10:                       # need 10 frames before the start
        nogo_trials.append(dff[start - 10 : start + 30])
nogo_trials = np.array(nogo_trials)
print(nogo_trials.shape)

## Solution — Step 7, 🔬 write it: `go_response` and `go_peak`

In [ ]:
go_response = go_trials.mean(axis=0)
go_peak = round(go_response[10:].max(), 3)
print(go_response.shape, "-> peak", go_peak)

## Solution — Step 8, 🔬 write it: the answer

In [ ]:
nogo_response = nogo_trials.mean(axis=0)
nogo_peak = round(nogo_response[10:].max(), 3)
ratio = round(go_peak / nogo_peak, 1)
print("go", go_peak, "| nogo", nogo_peak, "| go is", ratio, "times bigger")

## Solution — 🏆 Finale: `response_to`, and a second neuron

In [ ]:
def response_to(trace, marker):
    on = np.where(marker == 1)[0]
    starts = on[::9]
    trials = []
    for start in starts:
        if start >= 10:
            trials.append(trace[start - 10 : start + 30])
    trials = np.array(trials)
    return trials.mean(axis=0)

dff_2  = np.loadtxt(NEURON_2, delimiter=",", skiprows=1, usecols=1)
go_2   = np.loadtxt(NEURON_2, delimiter=",", skiprows=1, usecols=2)
nogo_2 = np.loadtxt(NEURON_2, delimiter=",", skiprows=1, usecols=3)

ratio_1 = round(response_to(dff, go)[10:].max() / response_to(dff, nogo)[10:].max(), 1)
ratio_2 = round(response_to(dff_2, go_2)[10:].max() / response_to(dff_2, nogo_2)[10:].max(), 1)
print("neuron 1: go is", ratio_1, "times nogo")
print("neuron 2: go is", ratio_2, "times nogo")